# **Improving Customer Retention through Payment Analytics**

In the **Olist** ecosystem, some customers pay in a single installment, while others use up to 24. The goal is to **analyze if payment installments and payment type (Credit Card vs. Voucher/Boleto) affect Customer Satisfaction (Review Scores) and Delivery Speed**.

## Imports

In [2]:
import kagglehub
import pandas as pd
import numpy as np
import os
from pathlib import Path

## **Download DataSet**

In [3]:
# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

DATA_DIR = Path(path)
print("Dataset downloaded to:", DATA_DIR)
print("\nFiles available:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(f"  {f.name}")

Dataset downloaded to: C:\Users\laura\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2

Files available:
  olist_customers_dataset.csv
  olist_geolocation_dataset.csv
  olist_order_items_dataset.csv
  olist_order_payments_dataset.csv
  olist_order_reviews_dataset.csv
  olist_orders_dataset.csv
  olist_products_dataset.csv
  olist_sellers_dataset.csv
  product_category_name_translation.csv


## **Data Strategy**

In [4]:
#Load individual tables

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv",
                     parse_dates=[
                         "order_purchase_timestamp",
                         "order_approved_at",
                         "order_delivered_carrier_date",
                         "order_delivered_customer_date",
                         "order_estimated_delivery_date"
                     ])
 
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
 
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv",
                      parse_dates=["review_creation_date", "review_answer_timestamp"])
 
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
 
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv",
                    parse_dates=["shipping_limit_date"])
 
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
 
category_translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")
 
print("\nTable shapes:")
for name, df in [("orders", orders), ("payments", payments), ("reviews", reviews),
                 ("customers", customers), ("items", items), ("products", products),
                 ("category_translation", category_translation)]:
    print(f"  {name:25s} {df.shape}")


Table shapes:
  orders                    (99441, 8)
  payments                  (103886, 5)
  reviews                   (99224, 7)
  customers                 (99441, 5)
  items                     (112650, 7)
  products                  (32951, 9)
  category_translation      (71, 2)


### Handling Missing Values

In [5]:
# Orders with missing delivery date are the ones not yet delivered — expected
print(f"\nOrders total:               {len(orders):,}")
print(f"Delivered orders:           {orders['order_delivered_customer_date'].notna().sum():,}")
print(f"Unique customers:           {orders['customer_id'].nunique():,}")
 
# Payments: some orders have multiple payment rows (split payments)
print(f"\nPayment rows:               {len(payments):,}")
print(f"Unique orders in payments:  {payments['order_id'].nunique():,}")
print(f"Payment types:\n{payments['payment_type'].value_counts()}")


Orders total:               99,441
Delivered orders:           96,476
Unique customers:           99,441

Payment rows:               103,886
Unique orders in payments:  99,440
Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


### Aggregate payments per order

- An order can be paid in multiple ways (e.g. voucher + credit card).
- Strategy: keep the dominant payment type (highest value), sum installments and total payment value across all payment rows for that order.

In [6]:
payments_agg = (
    payments
    .sort_values("payment_value", ascending=False)          # dominant type = highest value row
    .groupby("order_id")
    .agg(
        payment_type        = ("payment_type", "first"),    # dominant type
        payment_installments= ("payment_installments", "sum"),
        total_payment_value = ("payment_value", "sum"),
        n_payment_methods   = ("payment_type", "nunique")   # flag for split payments
    )
    .reset_index()
)
 
print(f"\nAggregated payment rows: {len(payments_agg):,}")
print(f"Split-payment orders:    {(payments_agg['n_payment_methods'] > 1).sum():,}")


Aggregated payment rows: 99,440
Split-payment orders:    2,246
